In [1]:
population_urls = {
    "Population": "https://www.globalfirepower.com/total-population-by-country.php",
    "Available_Manpower": "https://www.globalfirepower.com/available-military-manpower.php",
    "Fit_for_Service": "https://www.globalfirepower.com/manpower-fit-for-military-service.php",
    "Military_Age_Annually": "https://www.globalfirepower.com/manpower-reaching-military-age-annually.php",
    "Active_Personnel": "https://www.globalfirepower.com/active-military-manpower.php",
    "Reserve_Personnel": "https://www.globalfirepower.com/active-reserve-military-manpower.php",
    "Paramilitary": "https://www.globalfirepower.com/manpower-paramilitary.php"
}

In [2]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

In [4]:
def scrape_metric(url, metric_name):

    print(f"Scraping {metric_name}...")

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"Failed: {url}")
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    records = soup.find_all("div", class_="recordsetContainer")

    data = []

    for record in records:

        long_name = record.find("div", class_="longFormName")
        short_name = record.find("div", class_="shortFormName")
        value = record.find("div", class_="valueContainer")

        if value:

            if long_name:
                country = long_name.get_text(strip=True)
            elif short_name:
                country = short_name.get_text(strip=True)
            else:
                continue

            value = value.get_text(strip=True)

            data.append([country, value])

    df = pd.DataFrame(data, columns=["Country", metric_name])

    return df

In [5]:
dfs = []

for metric, url in population_urls.items():

    df = scrape_metric(url, metric)

    if df is not None:
        dfs.append(df)

    time.sleep(2)

Scraping Population...
Scraping Available_Manpower...
Scraping Fit_for_Service...
Scraping Military_Age_Annually...
Scraping Active_Personnel...
Scraping Reserve_Personnel...
Scraping Paramilitary...


In [6]:
population_df = dfs[0]

for df in dfs[1:]:

    population_df = population_df.merge(
        df,
        on="Country",
        how="outer"
    )

In [7]:
for col in population_df.columns[1:]:

    population_df[col] = (
        population_df[col]
        .str.replace(",", "", regex=False)
        .str.replace("+", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.strip()
    )

    population_df[col] = pd.to_numeric(
        population_df[col],
        errors="coerce"
    )

In [8]:
print(population_df.head())

       Country  Population  Available_Manpower  Fit_for_Service  \
0  Afghanistan    40121552            15647405          8826741   
1      Albania     3107100             1522479          1292554   
2      Algeria    47022473            22570787         19185169   
3       Angola    37202061             7440412          3720206   
4    Argentina    46994384            20677529         17575900   

   Military_Age_Annually  Active_Personnel  Reserve_Personnel  Paramilitary  
0                 842553             75000                  0         90000  
1                  62142              7500                  0           500  
2                 752360            130000             150000        150000  
3                 372021            107000                  0         10000  
4                 704916            108000              12370         20000  


In [9]:
population_df.to_csv("population_manpower.csv", index=False)

print("Population & Manpower dataset saved successfully!")

Population & Manpower dataset saved successfully!


In [10]:
print(population_df.shape)
print(population_df.info())
print(population_df.head(10))

(145, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Country                145 non-null    object
 1   Population             145 non-null    int64 
 2   Available_Manpower     145 non-null    int64 
 3   Fit_for_Service        145 non-null    int64 
 4   Military_Age_Annually  145 non-null    int64 
 5   Active_Personnel       145 non-null    int64 
 6   Reserve_Personnel      145 non-null    int64 
 7   Paramilitary           145 non-null    int64 
dtypes: int64(7), object(1)
memory usage: 9.2+ KB
None
       Country  Population  Available_Manpower  Fit_for_Service  \
0  Afghanistan    40121552            15647405          8826741   
1      Albania     3107100             1522479          1292554   
2      Algeria    47022473            22570787         19185169   
3       Angola    37202061             7440412        